# NeuroShield — Track 2: Brain Tumor Segmentation (BraTS2020, 2D nnU-Net)
### Master Pipeline — Corrected, Persistence-Safe Version

This notebook preserves the existing methodology exactly:
- Dataset: BraTS2020 (368 usable cases)
- Fixed seed = 42, outer split: Train 257 / Val 55 / Test 56
- Model A: 2D nnU-Net, `nnUNetTrainer_100epochs`, fold 0
- Outer 56-case test set stays untouched until Stage 8

**What changed vs. the previous notebook:**
- All artifacts now go under a clearly named, organized directory and are zipped for persistence
- `medpy` is completely removed — Dice/IoU/HD95 computed with scipy only (no dependency conflict)
- HD95 is computed in physical mm using NIfTI voxel spacing, not raw voxel counts
- Training cell checks for an existing checkpoint first, so re-running the notebook never accidentally retrains
- Explicit verification cells after training and after inference, before anything else proceeds

**How to run:** Run cells top to bottom, in order, **in one single interactive session**. Training, checkpoint packaging, inference, prediction packaging, metrics, and the final ZIP all happen in this same run — there is no second expensive step and no required re-commit afterward. Stop and read the printed output at Stage 7 and Stage 10 before continuing — those are checkpoints, not just narration. See Stage 13 for exactly how to persist the result without retraining.

## STAGE 1 — Environment & Path Setup

In [1]:
import os
import json
import shutil
import random
import datetime
import numpy as np
import pandas as pd
import nibabel as nib

# ---- Source dataset (existing, verified) ----
DATASET_PATH = "/kaggle/input/datasets/awsaf49/brats20-dataset-training-validation"

# ---- Working directories (organized under one named folder) ----
WORK_DIR = "/kaggle/working/neuroshield_track2"
NNUNET_RAW = f"{WORK_DIR}/nnUNet_raw"
NNUNET_PREPROCESSED = f"{WORK_DIR}/nnUNet_preprocessed"
NNUNET_RESULTS = f"{WORK_DIR}/nnUNet_results"
TEST_IMAGES = f"{WORK_DIR}/imagesTs"
TEST_PREDICTIONS = f"{WORK_DIR}/test_predictions"

# ---- Final, clearly-named persistent artifact directory ----
FINAL_ARTIFACT_DIR = "/kaggle/working/neuroshield_track2_model_a"
ARTIFACT_MODEL_DIR = f"{FINAL_ARTIFACT_DIR}/model"
ARTIFACT_PRED_DIR = f"{FINAL_ARTIFACT_DIR}/test_predictions"
ARTIFACT_METRICS_DIR = f"{FINAL_ARTIFACT_DIR}/metrics"
ARTIFACT_SPLITS_DIR = f"{FINAL_ARTIFACT_DIR}/splits"

DATASET_NAME = "Dataset001_BraTS2020"
DATASET_ID = "001"
TRAINER = "nnUNetTrainer_100epochs"
CONFIG = "2d"
FOLD = "0"
SEED = 42

for d in [NNUNET_RAW, NNUNET_PREPROCESSED, NNUNET_RESULTS, TEST_IMAGES, TEST_PREDICTIONS,
          ARTIFACT_MODEL_DIR, ARTIFACT_PRED_DIR, ARTIFACT_METRICS_DIR, ARTIFACT_SPLITS_DIR]:
    os.makedirs(d, exist_ok=True)

os.environ["nnUNet_raw"] = NNUNET_RAW
os.environ["nnUNet_preprocessed"] = NNUNET_PREPROCESSED
os.environ["nnUNet_results"] = NNUNET_RESULTS

print("Dataset path exists:", os.path.exists(DATASET_PATH))
print("nnUNet_raw          :", os.environ["nnUNet_raw"])
print("nnUNet_preprocessed :", os.environ["nnUNet_preprocessed"])
print("nnUNet_results      :", os.environ["nnUNet_results"])
print("Final artifact dir  :", FINAL_ARTIFACT_DIR)

Dataset path exists: True
nnUNet_raw          : /kaggle/working/neuroshield_track2/nnUNet_raw
nnUNet_preprocessed : /kaggle/working/neuroshield_track2/nnUNet_preprocessed
nnUNet_results      : /kaggle/working/neuroshield_track2/nnUNet_results
Final artifact dir  : /kaggle/working/neuroshield_track2_model_a


In [2]:
!pip install -q nnunetv2

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 291.1/291.1 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.8/102.8 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.8/76.8 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.6/53.6 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.1/28.1 MB 66.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 75.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 92.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 75.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.4/96.4 kB 8.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the 

## STAGE 2 — Dataset Discovery (unchanged from prior notebook)

In [3]:
TRAIN_DIR = os.path.join(DATASET_PATH, "BraTS2020_TrainingData", "MICCAI_BraTS2020_TrainingData")

cases = []
for entry in sorted(os.listdir(TRAIN_DIR)):
    case_path = os.path.join(TRAIN_DIR, entry)
    if not os.path.isdir(case_path):
        continue
    files = os.listdir(case_path)
    def find(suffix, files=files):
        matches = [f for f in files if f.lower().endswith(suffix)]
        return os.path.join(case_path, matches[0]) if matches else None

    row = {
        "case_id": entry,
        "flair": find("flair.nii"),
        "t1": find("t1.nii"),
        "t1ce": find("t1ce.nii"),
        "t2": find("t2.nii"),
        "seg": find("seg.nii"),
    }
    if all(row.values()):
        cases.append(row)

print(f"Usable cases with all 4 modalities + mask: {len(cases)}")
assert len(cases) == 368, f"Expected 368 usable cases, found {len(cases)} — dataset mount may differ."

Usable cases with all 4 modalities + mask: 368


## STAGE 3 — Fixed Outer Split (seed=42, 257/55/56 — unchanged)

In [4]:
random.seed(SEED)

case_ids = [c["case_id"] for c in cases]
random.shuffle(case_ids)

n = len(case_ids)
n_train = int(0.70 * n)
n_val = int(0.15 * n)

train_ids = set(case_ids[:n_train])
val_ids = set(case_ids[n_train:n_train + n_val])
test_ids = set(case_ids[n_train + n_val:])

print(f"Train: {len(train_ids)}  Val: {len(val_ids)}  Test: {len(test_ids)}")
assert len(train_ids) == 257 and len(val_ids) == 55 and len(test_ids) == 56, \
    "Split counts do not match the established 257/55/56 outer split."

assert not (train_ids & val_ids)
assert not (val_ids & test_ids)
assert not (train_ids & test_ids)
print("No overlap between train/val/test — confirmed.")

Train: 257  Val: 55  Test: 56
No overlap between train/val/test — confirmed.


In [5]:
df = pd.DataFrame(cases)

def assign_split(cid):
    if cid in train_ids: return "train"
    if cid in val_ids: return "val"
    return "test"

df["split"] = df["case_id"].apply(assign_split)

manifest_path = os.path.join(ARTIFACT_SPLITS_DIR, "braintumor_manifest.csv")
df.to_csv(manifest_path, index=False)

print(df["split"].value_counts())
print("Manifest saved to:", manifest_path)

split
train    257
test      56
val       55
Name: count, dtype: int64
Manifest saved to: /kaggle/working/neuroshield_track2_model_a/splits/braintumor_manifest.csv


## STAGE 4 — nnU-Net Dataset Preparation (conversion + dataset.json)

In [6]:
RAW_BASE = os.path.join(NNUNET_RAW, DATASET_NAME)
imagesTr = os.path.join(RAW_BASE, "imagesTr")
labelsTr = os.path.join(RAW_BASE, "labelsTr")
os.makedirs(imagesTr, exist_ok=True)
os.makedirs(labelsTr, exist_ok=True)

trainval = df[df["split"].isin(["train", "val"])].reset_index(drop=True)
test_cases = df[df["split"] == "test"].reset_index(drop=True)
print(f"Cases going into nnU-Net's training pool: {len(trainval)}")
print(f"Outer test cases (untouched until Stage 8): {len(test_cases)}")
assert len(test_cases) == 56, "Outer test set must contain exactly 56 cases."

Cases going into nnU-Net's training pool: 312
Outer test cases (untouched until Stage 8): 56


In [7]:
modality_map = {"flair": "0000", "t1": "0001", "t1ce": "0002", "t2": "0003"}

# Skip conversion if already done (avoids redundant compute on re-run)
already_converted = os.path.exists(imagesTr) and len(os.listdir(imagesTr)) >= len(trainval) * 4

if already_converted:
    print("imagesTr/labelsTr already populated — skipping conversion.")
else:
    for i, row in trainval.iterrows():
        case_id = row["case_id"]
        for mod, channel in modality_map.items():
            img = nib.load(row[mod])
            nib.save(img, os.path.join(imagesTr, f"{case_id}_{channel}.nii.gz"))

        seg_img = nib.load(row["seg"])
        seg_data = seg_img.get_fdata()
        seg_data[seg_data == 4] = 3  # nnU-Net needs consecutive labels: 0,1,2,3
        remapped = nib.Nifti1Image(seg_data.astype(np.uint8), seg_img.affine, seg_img.header)
        nib.save(remapped, os.path.join(labelsTr, f"{case_id}.nii.gz"))

        if (i + 1) % 50 == 0:
            print(f"Converted {i+1}/{len(trainval)} cases")
    print("Conversion complete.")

Converted 50/312 cases
Converted 100/312 cases
Converted 150/312 cases
Converted 200/312 cases
Converted 250/312 cases
Converted 300/312 cases
Conversion complete.


In [8]:
# Verify one converted case
sample_id = trainval.iloc[0]["case_id"]
print("Image files:", [f for f in sorted(os.listdir(imagesTr)) if f.startswith(sample_id)])
sample_label = nib.load(os.path.join(labelsTr, f"{sample_id}.nii.gz")).get_fdata()
print("Label values after remap:", np.unique(sample_label))  # expect [0. 1. 2. 3.]

Image files: ['BraTS20_Training_001_0000.nii.gz', 'BraTS20_Training_001_0001.nii.gz', 'BraTS20_Training_001_0002.nii.gz', 'BraTS20_Training_001_0003.nii.gz']
Label values after remap: [0. 1. 2. 3.]


In [9]:
dataset_json = {
    "channel_names": {"0": "FLAIR", "1": "T1", "2": "T1CE", "3": "T2"},
    "labels": {
        "background": 0,
        "necrotic_non_enhancing_core": 1,
        "edema": 2,
        "enhancing_tumor": 3
    },
    "numTraining": len(trainval),
    "file_ending": ".nii.gz"
}

with open(os.path.join(RAW_BASE, "dataset.json"), "w") as f:
    json.dump(dataset_json, f, indent=4)

print(json.dumps(dataset_json, indent=4))

{
    "channel_names": {
        "0": "FLAIR",
        "1": "T1",
        "2": "T1CE",
        "3": "T2"
    },
    "labels": {
        "background": 0,
        "necrotic_non_enhancing_core": 1,
        "edema": 2,
        "enhancing_tumor": 3
    },
    "numTraining": 312,
    "file_ending": ".nii.gz"
}


## STAGE 5 — Preprocessing

In [10]:
# Skip if already preprocessed (avoids redundant compute on re-run)
preprocessed_marker = os.path.join(NNUNET_PREPROCESSED, DATASET_NAME)
if os.path.exists(preprocessed_marker) and len(os.listdir(preprocessed_marker)) > 0:
    print("Preprocessed data already exists — skipping plan_and_preprocess.")
else:
    !nnUNetv2_plan_and_preprocess -d {DATASET_ID} --verify_dataset_integrity

Fingerprint extraction...
Dataset001_BraTS2020
Using <class 'nnunetv2.imageio.simpleitk_reader_writer.SimpleITKIO'> as reader/writer

####################
verify_dataset_integrity Done. 
If you didn't see any error messages then your dataset is most likely OK!
####################

Using <class 'nnunetv2.imageio.simpleitk_reader_writer.SimpleITKIO'> as reader/writer
Extracting dataset fingerprint: 100%|█████████| 312/312 [01:57<00:00,  2.66it/s]
Experiment planning...

############################
INFO: You are using the old nnU-Net default planner. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

Dropping 3d_lowres config because the image size difference to 3d_fullres is too small. 3d_fullres: [139. 169. 138.], 3d_lowres: [139, 169, 138]
2D U-Net configuration:
{'data_identifier': 'nnUNetPlans_2d', 'preprocessor_name': 'DefaultPreproces

In [11]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: GPU is NOT available")

PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


In [12]:
import os
import glob

print("=== SPLIT / MODEL SAFETY CHECK ===")

for root in [
    "/kaggle/working/neuroshield_track2",
    "/kaggle/working/neuroshield_track2_model_a",
]:
    print(f"\n{root}")
    if os.path.exists(root):
        for path in glob.glob(root + "/**/*", recursive=True):
            if os.path.isfile(path) and (
                "manifest" in os.path.basename(path).lower()
                or "checkpoint" in os.path.basename(path).lower()
            ):
                print(path)
    else:
        print("Directory does not exist")

print("\n=== CHECK COMPLETE ===")

=== SPLIT / MODEL SAFETY CHECK ===

/kaggle/working/neuroshield_track2

/kaggle/working/neuroshield_track2_model_a
/kaggle/working/neuroshield_track2_model_a/splits/braintumor_manifest.csv

=== CHECK COMPLETE ===


In [13]:
import os
import pandas as pd

manifest = "/kaggle/working/neuroshield_track2_model_a/splits/braintumor_manifest.csv"

df = pd.read_csv(manifest)

print("=== FINAL PRE-TRAINING CHECK ===")
print("Manifest:", manifest)
print("Train:", (df["split"] == "train").sum())
print("Val:", (df["split"] == "val").sum())
print("Test:", (df["split"] == "test").sum())

# Safety check: existing trained checkpoint
results_dir = "/kaggle/working/neuroshield_track2/nnUNet_results"

checkpoints = []
if os.path.exists(results_dir):
    for root, dirs, files in os.walk(results_dir):
        for f in files:
            if f in ["checkpoint_final.pth", "checkpoint_best.pth"]:
                checkpoints.append(os.path.join(root, f))

print("\nExisting checkpoints:")
if checkpoints:
    for p in checkpoints:
        print("FOUND:", p)
else:
    print("None found")

assert (df["split"] == "train").sum() == 257
assert (df["split"] == "val").sum() == 55
assert (df["split"] == "test").sum() == 56

print("\n=== SAFE TO PROCEED ===")

=== FINAL PRE-TRAINING CHECK ===
Manifest: /kaggle/working/neuroshield_track2_model_a/splits/braintumor_manifest.csv
Train: 257
Val: 55
Test: 56

Existing checkpoints:
None found

=== SAFE TO PROCEED ===


## STAGE 6 — Model A Training
**Safety check built in:** this cell checks for an existing final checkpoint before training. If found, training is skipped entirely — re-running this notebook will never silently retrain and burn GPU hours. Delete the checkpoint file manually if you deliberately want to retrain.

In [14]:
expected_ckpt_dir = os.path.join(
    NNUNET_RESULTS, DATASET_NAME,
    f"{TRAINER}__nnUNetPlans__{CONFIG}", f"fold_{FOLD}"
)
final_ckpt = os.path.join(expected_ckpt_dir, "checkpoint_final.pth")

if os.path.exists(final_ckpt):
    print("Final checkpoint already exists — SKIPPING training to avoid accidental retrain:")
    print(final_ckpt)
else:
    print("No final checkpoint found. Starting/resuming training.")
    print("Note: '--c' is safe to use even in a fresh environment — nnU-Net will simply report")
    print("'no checkpoint available, starting new training' and begin cleanly. It only resumes")
    print("from a checkpoint that matches this exact dataset/trainer/fold, never an unrelated one.")
    !nnUNetv2_train {DATASET_ID} {CONFIG} {FOLD} -tr {TRAINER} --c

No final checkpoint found. Starting/resuming training.
Note: '--c' is safe to use even in a fresh environment — nnU-Net will simply report
'no checkpoint available, starting new training' and begin cleanly. It only resumes
from a checkpoint that matches this exact dataset/trainer/fold, never an unrelated one.

############################
INFO: You are using the old nnU-Net default plans. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

Using device: cuda:0

#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
##########################################################

## STAGE 7 — CHECKPOINT VERIFICATION (stop and check before continuing)


In [15]:
config_dir = os.path.dirname(expected_ckpt_dir)  # the .../nnUNetTrainer_100epochs__nnUNetPlans__2d/ level

best_ckpt = os.path.join(expected_ckpt_dir, "checkpoint_best.pth")
plans_json = os.path.join(config_dir, "plans.json")
ds_json = os.path.join(config_dir, "dataset.json")

required_files = {
    "checkpoint_final.pth": final_ckpt,
    "checkpoint_best.pth": best_ckpt,
    "plans.json": plans_json,
    "dataset.json": ds_json,
}

all_ok = True
for name, path in required_files.items():
    exists = os.path.exists(path)
    size_mb = os.path.getsize(path) / 1e6 if exists else 0
    status = "OK" if exists else "MISSING"
    print(f"[{status}] {name:<22} {size_mb:8.2f} MB   {path}")
    if not exists:
        all_ok = False

assert all_ok, "One or more required checkpoint/config files are missing. Do not proceed to Stage 8."
print("\nAll required checkpoint files verified. Safe to proceed to Stage 8.")

[OK] checkpoint_final.pth     165.14 MB   /kaggle/working/neuroshield_track2/nnUNet_results/Dataset001_BraTS2020/nnUNetTrainer_100epochs__nnUNetPlans__2d/fold_0/checkpoint_final.pth
[OK] checkpoint_best.pth      165.12 MB   /kaggle/working/neuroshield_track2/nnUNet_results/Dataset001_BraTS2020/nnUNetTrainer_100epochs__nnUNetPlans__2d/fold_0/checkpoint_best.pth
[OK] plans.json                 0.01 MB   /kaggle/working/neuroshield_track2/nnUNet_results/Dataset001_BraTS2020/nnUNetTrainer_100epochs__nnUNetPlans__2d/plans.json
[OK] dataset.json               0.00 MB   /kaggle/working/neuroshield_track2/nnUNet_results/Dataset001_BraTS2020/nnUNetTrainer_100epochs__nnUNetPlans__2d/dataset.json

All required checkpoint files verified. Safe to proceed to Stage 8.


## STAGE 8 — BACKUP / PERSIST MODEL
This runs immediately after training in the **same session** — no separate commit/re-run needed. The saved structure preserves nnU-Net's own required layout (`model/Dataset001_BraTS2020/<trainer>__nnUNetPlans__<config>/fold_X/`), so it can be pointed to directly as `nnUNet_results` in a future session for inference, with no reshaping needed.

In [16]:
artifact_dataset_dir = os.path.join(ARTIFACT_MODEL_DIR, DATASET_NAME)
artifact_config_dir = os.path.join(artifact_dataset_dir, f"{TRAINER}__nnUNetPlans__{CONFIG}")
artifact_fold_dir = os.path.join(artifact_config_dir, f"fold_{FOLD}")
os.makedirs(artifact_config_dir, exist_ok=True)

if os.path.exists(artifact_fold_dir):
    shutil.rmtree(artifact_fold_dir)
shutil.copytree(expected_ckpt_dir, artifact_fold_dir)

# plans.json and dataset.json live at the config level, one up from fold_0
shutil.copy(plans_json, os.path.join(artifact_config_dir, "plans.json"))
shutil.copy(ds_json, os.path.join(artifact_config_dir, "dataset.json"))

print("Model artifact copied to:", artifact_fold_dir)
print("Full nnU-Net-compatible structure:")
for root, dirs, files in os.walk(ARTIFACT_MODEL_DIR):
    level = root.replace(ARTIFACT_MODEL_DIR, "").count(os.sep)
    print("  " * level + os.path.basename(root) + "/")
    for f in files:
        print("  " * (level + 1) + f)

Model artifact copied to: /kaggle/working/neuroshield_track2_model_a/model/Dataset001_BraTS2020/nnUNetTrainer_100epochs__nnUNetPlans__2d/fold_0
Full nnU-Net-compatible structure:
model/
  Dataset001_BraTS2020/
    nnUNetTrainer_100epochs__nnUNetPlans__2d/
      dataset.json
      plans.json
      fold_0/
        checkpoint_best.pth
        training_log_2026_9_24_18_06_01.txt
        debug.json
        checkpoint_final.pth
        progress.png
        validation/
          BraTS20_Training_305.nii.gz
          BraTS20_Training_085.nii.gz
          BraTS20_Training_169.nii.gz
          BraTS20_Training_299.nii.gz
          BraTS20_Training_354.nii.gz
          BraTS20_Training_140.nii.gz
          BraTS20_Training_158.nii.gz
          BraTS20_Training_043.nii.gz
          BraTS20_Training_250.nii.gz
          BraTS20_Training_090.nii.gz
          BraTS20_Training_037.nii.gz
          summary.json
          BraTS20_Training_252.nii.gz
          BraTS20_Training_274.nii.gz
          BraTS2

In [17]:
training_metadata = {
    "project": "NeuroShield",
    "track": "Track 2 - Brain Tumor Segmentation",
    "dataset": "BraTS2020",
    "total_usable_cases": len(cases),
    "train_count": len(train_ids),
    "val_count": len(val_ids),
    "test_count": len(test_ids),
    "seed": SEED,
    "model": "2D nnU-Net (Model A)",
    "trainer": TRAINER,
    "configuration": CONFIG,
    "fold": FOLD,
    "epochs": 100,
    "run_datetime_utc": datetime.datetime.utcnow().isoformat()
}

with open(os.path.join(FINAL_ARTIFACT_DIR, "training_metadata.json"), "w") as f:
    json.dump(training_metadata, f, indent=4)

print(json.dumps(training_metadata, indent=4))

{
    "project": "NeuroShield",
    "track": "Track 2 - Brain Tumor Segmentation",
    "dataset": "BraTS2020",
    "total_usable_cases": 368,
    "train_count": 257,
    "val_count": 55,
    "test_count": 56,
    "seed": 42,
    "model": "2D nnU-Net (Model A)",
    "trainer": "nnUNetTrainer_100epochs",
    "configuration": "2d",
    "fold": "0",
    "epochs": 100,
    "run_datetime_utc": "2026-09-24T22:41:07.888042"
}


/tmp/ipykernel_23/3161550400.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "run_datetime_utc": datetime.datetime.utcnow().isoformat()


## STAGE 9 — Outer Test Inference (56-case untouched set)

In [18]:
# Convert test images (no labels needed for inference) — skip if already done
already_converted_ts = os.path.exists(TEST_IMAGES) and len(os.listdir(TEST_IMAGES)) >= len(test_cases) * 4

if already_converted_ts:
    print("Test images already converted — skipping.")
else:
    for i, row in test_cases.iterrows():
        case_id = row["case_id"]
        for mod, channel in modality_map.items():
            img = nib.load(row[mod])
            nib.save(img, os.path.join(TEST_IMAGES, f"{case_id}_{channel}.nii.gz"))
    print("Test images converted.")

Test images converted.


In [19]:
existing_preds = [f for f in os.listdir(TEST_PREDICTIONS) if f.endswith('.nii.gz')] if os.path.exists(TEST_PREDICTIONS) else []

if len(existing_preds) >= len(test_cases):
    print(f"Found {len(existing_preds)} existing predictions — skipping inference.")
else:
    !nnUNetv2_predict -i {TEST_IMAGES} -o {TEST_PREDICTIONS} -d {DATASET_ID} -c {CONFIG} -f {FOLD} -tr {TRAINER}


#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################

There are 56 cases in the source folder
I am process 0 out of 1 (max process ID is 0, we start counting with 0!)
There are 56 cases that I would like to predict

Predicting BraTS20_Training_004:
perform_everything_on_device: True
100%|█████████████████████████████████████████| 140/140 [00:05<00:00, 24.41it/s]
sending off prediction to background worker for resampling and export
done with BraTS20_Training_004

Predicting BraTS20_Training_013:
perform_everything_on_device: True
100%|█████████████████████████████████████████| 140/140 [00:04<00:00, 30.96it/s]
sending off prediction to backgroun

## STAGE 10 — TEST PREDICTION VERIFICATION (stop and check before continuing)

In [20]:
pred_files = [f for f in os.listdir(TEST_PREDICTIONS) if f.endswith(".nii.gz")]
print(f"Predictions found: {len(pred_files)} / {len(test_cases)}")
assert len(pred_files) == len(test_cases) == 56, "Prediction count does not match the 56-case outer test set."

pred_ids = set(f.replace(".nii.gz", "") for f in pred_files)
test_id_set = set(test_cases["case_id"])
missing = test_id_set - pred_ids
assert not missing, f"Missing predictions for: {missing}"

# Confirm zero overlap between outer test set and train/val (integrity check)
assert not (test_id_set & train_ids), "Test set overlaps with training set!"
assert not (test_id_set & val_ids), "Test set overlaps with validation set!"

print("All 56 outer test cases have predictions.")
print("Confirmed: outer test set has zero overlap with train/val.")
print("Safe to proceed to Stage 11.")

Predictions found: 56 / 56
All 56 outer test cases have predictions.
Confirmed: outer test set has zero overlap with train/val.
Safe to proceed to Stage 11.


## STAGE 11 — BACKUP / PERSIST PREDICTIONS

In [21]:
if os.path.exists(ARTIFACT_PRED_DIR):
    shutil.rmtree(ARTIFACT_PRED_DIR)
shutil.copytree(TEST_PREDICTIONS, ARTIFACT_PRED_DIR)

print("Predictions copied to:", ARTIFACT_PRED_DIR)
print("File count:", len([f for f in os.listdir(ARTIFACT_PRED_DIR) if f.endswith('.nii.gz')]))

Predictions copied to: /kaggle/working/neuroshield_track2_model_a/test_predictions
File count: 56


## STAGE 12 — Final Test Metrics (Dice, IoU, HD95 — no MedPy)
Computed against the original 56-case outer test ground truth (`test_cases`), never nnU-Net's internal validation fold. HD95 uses voxel spacing from each NIfTI's affine, so it is reported in millimeters, not raw voxel counts.

In [22]:
from scipy.ndimage import distance_transform_edt

def dice_score(pred, gt):
    intersection = np.sum(pred * gt)
    return (2.0 * intersection) / (pred.sum() + gt.sum())

def iou_score(pred, gt):
    intersection = np.sum(pred * gt)
    union = np.sum(pred) + np.sum(gt) - intersection
    return intersection / union if union > 0 else 1.0

def hd95_mm(pred, gt, spacing):
    """95th percentile Hausdorff Distance in physical mm, using voxel spacing."""
    if pred.sum() == 0 or gt.sum() == 0:
        return np.nan
    pred_dt = distance_transform_edt(~pred.astype(bool), sampling=spacing)
    gt_dt = distance_transform_edt(~gt.astype(bool), sampling=spacing)
    pred_to_gt = pred_dt[gt.astype(bool)]
    gt_to_pred = gt_dt[pred.astype(bool)]
    return max(np.percentile(pred_to_gt, 95), np.percentile(gt_to_pred, 95))

def get_region_masks(label_array):
    wt = (label_array > 0).astype(np.uint8)
    tc = np.isin(label_array, [1, 3]).astype(np.uint8)
    et = (label_array == 3).astype(np.uint8)
    return wt, tc, et

ImportError: cannot import name '_center' from 'numpy._core.umath' (/usr/local/lib/python3.12/dist-packages/numpy/_core/umath.py)

In [ ]:
results = []

for i, row in test_cases.iterrows():
    case_id = row["case_id"]

    gt_img = nib.load(row["seg"])
    spacing = gt_img.header.get_zooms()[:3]
    gt = gt_img.get_fdata()
    gt[gt == 4] = 3
    gt = gt.astype(np.uint8)

    pred_path = os.path.join(TEST_PREDICTIONS, f"{case_id}.nii.gz")
    pred = nib.load(pred_path).get_fdata().astype(np.uint8)

    gt_wt, gt_tc, gt_et = get_region_masks(gt)
    pred_wt, pred_tc, pred_et = get_region_masks(pred)

    row_result = {"case_id": case_id}
    for name, (g, p) in zip(["WT", "TC", "ET"],
                             [(gt_wt, pred_wt), (gt_tc, pred_tc), (gt_et, pred_et)]):
        if g.sum() == 0 and p.sum() == 0:
            row_result[f"{name}_dice"], row_result[f"{name}_iou"], row_result[f"{name}_hd95_mm"] = 1.0, 1.0, 0.0
        elif g.sum() == 0 or p.sum() == 0:
            row_result[f"{name}_dice"], row_result[f"{name}_iou"], row_result[f"{name}_hd95_mm"] = 0.0, 0.0, np.nan
        else:
            row_result[f"{name}_dice"] = dice_score(p, g)
            row_result[f"{name}_iou"] = iou_score(p, g)
            row_result[f"{name}_hd95_mm"] = hd95_mm(p, g, spacing)

    results.append(row_result)
    if (i + 1) % 10 == 0:
        print(f"Evaluated {i+1}/{len(test_cases)}")

print("Evaluation complete.")

In [ ]:
results_df = pd.DataFrame(results)
metrics_path = os.path.join(ARTIFACT_METRICS_DIR, "model_a_test_metrics.csv")
results_df.to_csv(metrics_path, index=False)

summary = results_df[["WT_dice", "TC_dice", "ET_dice",
                       "WT_iou", "TC_iou", "ET_iou",
                       "WT_hd95_mm", "TC_hd95_mm", "ET_hd95_mm"]].mean()

print("Mean metrics across 56 outer test cases:\n")
print(summary)
print("\nSaved to:", metrics_path)

## STAGE 13 — Final Artifact Inventory, Archive, and Download Instructions

In [ ]:
readme_text = f"""NeuroShield — Track 2: Brain Tumor Segmentation
Model A: 2D nnU-Net (trainer={TRAINER}, config={CONFIG}, fold={FOLD})
Dataset: BraTS2020, 368 usable cases, seed={SEED}
Split: Train={len(train_ids)}  Val={len(val_ids)}  Test={len(test_ids)}

CONTENTS:
  model/                       trained checkpoints + plans.json + dataset.json for this fold
  test_predictions/            56 predicted segmentation masks (.nii.gz) for the outer test set
  metrics/model_a_test_metrics.csv   per-case Dice / IoU / HD95(mm) for WT, TC, ET regions
  splits/braintumor_manifest.csv     the exact case_id -> split assignment used
  training_metadata.json       run configuration and date

REUSE: to run inference on a new scan later, set nnUNet_results to point at the 'model/' directory
structure above (it must remain nested as Dataset001_BraTS2020/<trainer>__nnUNetPlans__<config>/fold_X/)
and call nnUNetv2_predict as in Stage 9.
"""

with open(os.path.join(FINAL_ARTIFACT_DIR, "README.txt"), "w") as f:
    f.write(readme_text)

print(readme_text)

In [ ]:
# Verify full artifact tree exists before archiving
def dir_size_mb(path):
    total = 0
    for root, _, files in os.walk(path):
        for f in files:
            total += os.path.getsize(os.path.join(root, f))
    return total / 1e6

print("FINAL ARTIFACT INVENTORY")
print("=" * 50)
for sub in ["model", "test_predictions", "metrics", "splits"]:
    p = os.path.join(FINAL_ARTIFACT_DIR, sub)
    exists = os.path.exists(p)
    size = dir_size_mb(p) if exists else 0
    n_files = sum(len(files) for _, _, files in os.walk(p)) if exists else 0
    print(f"{sub:<18} exists={exists}  files={n_files:<4} size={size:8.2f} MB")

assert os.path.exists(os.path.join(artifact_fold_dir, 'checkpoint_best.pth'))
assert os.path.exists(os.path.join(artifact_fold_dir, 'checkpoint_final.pth'))
assert os.path.exists(os.path.join(artifact_config_dir, 'plans.json'))
assert os.path.exists(os.path.join(artifact_config_dir, 'dataset.json'))
assert len([f for f in os.listdir(ARTIFACT_PRED_DIR) if f.endswith('.nii.gz')]) == 56
assert os.path.exists(os.path.join(ARTIFACT_METRICS_DIR, 'model_a_test_metrics.csv'))
print("\nAll required artifacts verified present, in the correct nnU-Net-compatible structure.")

In [ ]:
archive_path = shutil.make_archive(FINAL_ARTIFACT_DIR, 'zip', FINAL_ARTIFACT_DIR)
archive_size_mb = os.path.getsize(archive_path) / 1e6
print(f"Archive created: {archive_path}  ({archive_size_mb:.2f} MB)")

In [ ]:
# Verify the ZIP itself actually contains the real .pth weights (not just empty folders)
import zipfile

with zipfile.ZipFile(archive_path, 'r') as zf:
    names = zf.namelist()
    pth_entries = [n for n in names if n.endswith('.pth')]
    print(f"Total files in archive: {len(names)}")
    print(f".pth weight files found in archive: {len(pth_entries)}")
    for n in pth_entries:
        info = zf.getinfo(n)
        print(f"  {n}   ({info.file_size / 1e6:.2f} MB)")

assert len(pth_entries) >= 2, "ZIP does not contain the expected checkpoint_best.pth and checkpoint_final.pth!"
assert any('checkpoint_best.pth' in n for n in pth_entries), "checkpoint_best.pth missing from archive!"
assert any('checkpoint_final.pth' in n for n in pth_entries), "checkpoint_final.pth missing from archive!"
print("\nConfirmed: ZIP contains the real trained model weights.")

### IMPORTANT — how to actually persist this, without retraining
`/kaggle/working/neuroshield_track2_model_a.zip` exists only on this session's disk right now. **Everything in this notebook — training, packaging, and the ZIP — has already happened in this one session.** Do NOT click "Save & Run All (Commit)" — that starts a brand-new session with an empty `/kaggle/working`, which would trigger the 100-epoch training cell to run again.

Instead, do ONE of these, using what is **already sitting in this session's memory/disk**:

1. **Recommended — Quick Save:** click **Save Version** → choose **"Quick Save"** (not "Save & Run All").    Quick Save snapshots the current session's `/kaggle/working/` files — including the ZIP — as this    version's Output, **without re-executing any cell**. This is the safe option.
2. **Alternative — direct download:** use the file browser panel on the right → download    `neuroshield_track2_model_a.zip` straight to your computer right now, before the session ends.
3. **For reuse across notebooks:** after a Quick Save, open that version's **Output** tab →    **"New Dataset"** from the output, so Model B/C notebooks can attach this trained Model A as an input    without ever retraining it.

Do this now, in this session, before it closes or times out.

In [ ]:
print("=" * 50)
print("NEUROSHIELD TRACK 2 -- MODEL A ARTIFACT")
print("=" * 50)
print(f"Model: 2D nnU-Net ({TRAINER})")
print(f"Dataset: BraTS2020")
print(f"Cases: {len(cases)}  Train: {len(train_ids)}  Val: {len(val_ids)}  Outer Test: {len(test_ids)}")
print(f"Epochs: 100  Fold: {FOLD}")
print()
print("BEST CHECKPOINT   :", os.path.join(artifact_fold_dir, 'checkpoint_best.pth'))
print("FINAL CHECKPOINT  :", os.path.join(artifact_fold_dir, 'checkpoint_final.pth'))
print("TEST PREDICTIONS  : 56 / 56")
print("METRICS (mean)    :")
print(summary.to_string())
print("ARCHIVE           :", archive_path)
print()
print("=" * 50)
print("MODEL A IS READY FOR FUTURE INFERENCE")
print("=" * 50)


```python
# In a NEW notebook session, after attaching the saved output/dataset as input:
import os
os.environ["nnUNet_raw"] = "/kaggle/working/nnUNet_raw"            # only needs a valid path, unused for inference-only
os.environ["nnUNet_preprocessed"] = "/kaggle/working/nnUNet_preprocessed"
os.environ["nnUNet_results"] = "/kaggle/input/<your-saved-dataset-name>/model"  # points at the model/ folder itself

# new_scan_dir must contain files named <case_id>_0000.nii.gz .. _0003.nii.gz (FLAIR,T1,T1CE,T2)
!nnUNetv2_predict -i /path/to/new_scan_dir -o /path/to/output_dir \
    -d 001 -c 2d -f 0 -tr nnUNetTrainer_100epochs
```